Raw Memory, Strides & C-Types (The Direct Triton/CUDA Bridge)

Triton, CUDA, and C++ kernels do not work with high-level Python objects. They operate directly on raw memory pointers, byte offsets, and stride arithmetic.

1. Strides and The General Offset FormulaWhen a multi-dimensional tensor is flattened into 1D memory, strides define how many elements (or bytes) you must jump in physical memory to move 1 step along each dimension.Row-Major (C-Style, Default in Python/PyTorch/C++): Last dimension is contiguous (stride = 1).Formula for any N-D index $(i_0, i_1, \dots, i_{n-1})$:$$\text{Memory Offset} = \sum_{k=0}^{n-1} (i_k \times \text{stride}_k)$$For a 3D Tensor of shape $(D, H, W)$:$$\text{Offset}(d, h, w) = d \times (H \times W) + h \times W + w \times 1$$In Triton kernels, you write this stride math explicitly to load block pointers from global memory:tl.load(ptr + offsets_row[:, None] * stride_r + offsets_col[None, :] * stride_c).

1. Strides = “how far do I jump?”

Suppose:

A B C
D E F

Memory is actually:

[A, B, C, D, E, F]

Each element has a position:

A=0  B=1  C=2  D=3  E=4  F=5

A stride tells you:

“If I move 1 step in this dimension, how many memory elements do I jump?”

For this 2×3 array:

shape   = (2, 3)
strides = (3, 1)

Why?

Move one row down → jump 3 elements → stride_row = 3
Move one column right → jump 1 element → stride_col = 1

So:

offset = row * stride_row + col * stride_col

For E:

offset = 1 * 3 + 1 * 1
       = 4

So E lives at buffer[4].